# 10. Обучение Span-based RuBERT NER

Модель обучается на `corrected_v1` минимум 15 и максимум 20 эпох. Early stopping включается только после 15-й эпохи. Все промежуточные сохранения выполняются на локальном диске Colab (`/content`), поэтому Google Drive не накапливает версии файла весов. После обучения в Drive один раз копируется только итоговый run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import runpy

PROJECT_DIR = Path('/content/drive/MyDrive/NER_RuREBus_project')
EXPERIMENT_CONFIG = PROJECT_DIR / 'configs/experiments/span_ner_corrected_v1.yaml'
BOOTSTRAP = PROJECT_DIR / 'colab_bootstrap.py'
RUNTIME_OUTPUT = Path('/content/rurebus_runs/span_ner_corrected_v1/seed_42')
DRIVE_OUTPUT = PROJECT_DIR / 'results/span_ner_corrected_v1/seed_42'

for required_path in (EXPERIMENT_CONFIG, BOOTSTRAP):
    if not required_path.is_file():
        raise FileNotFoundError(f'Не найден {required_path}. Обновите проект в Google Drive.')

bootstrap_project = runpy.run_path(str(BOOTSTRAP))['bootstrap_project']
bootstrap_project(PROJECT_DIR)

In [ ]:
import torch

print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Для Span NER выберите GPU runtime в Colab.')

In [ ]:
from rurebus_ie.training import train_span_ner_experiment

summary = train_span_ner_experiment(
    EXPERIMENT_CONFIG,
    project_root=PROJECT_DIR,
    output_dir_override=RUNTIME_OUTPUT,
)
print(f'Лучшая эпоха: {summary.best_epoch}')
print(f'Validation strict micro-F1: {summary.best_validation_f1:.4f}')
print(f'Локальный checkpoint: {summary.checkpoint_dir}')

In [ ]:
import pandas as pd

history = pd.DataFrame(summary.history)
display(history)
history.plot(x='epoch', y=['train_loss', 'validation_loss'], grid=True);
history.plot(x='epoch', y=['validation_micro_f1', 'validation_macro_f1'], ylim=(0, 1), grid=True);

## Экспорт одного итогового checkpoint в Drive

Ячейка намеренно не перезаписывает существующий run: это защищает Drive от повторного накопления версий больших весов.

In [ ]:
import shutil

if DRIVE_OUTPUT.exists():
    raise FileExistsError(
        f'{DRIVE_OUTPUT} уже существует. Не перезаписывайте веса: задайте новый run/seed или удалите старый осознанно.'
    )
DRIVE_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(RUNTIME_OUTPUT, DRIVE_OUTPUT)
print('Итоговый run сохранён один раз:', DRIVE_OUTPUT)